# CUDA Programming Basics

## Historical Context

**CUDA (Compute Unified Device Architecture)** was introduced by NVIDIA in 2006, revolutionizing GPU computing:

### Timeline
- **2006**: CUDA 1.0 released with Tesla architecture
- **2008**: CUDA 2.0 adds shared memory atomics
- **2010**: Fermat architecture introduces ECC memory, L1/L2 cache
- **2012**: Kepler architecture adds dynamic parallelism
- **2014**: Maxwell architecture focuses on energy efficiency
- **2016**: Pascal architecture introduces unified memory
- **2017**: Volta introduces Tensor Cores for ML
- **2018**: Turing adds RT cores for ray tracing
- **2020**: Ampere improves Tensor Core performance
- **2022**: Hopper (H100) with Transformer Engine

### Why CUDA Matters for Deep Learning
- **Massive Parallelism**: Thousands of threads executing simultaneously
- **Memory Hierarchy**: Optimize for bandwidth vs latency
- **Custom Kernels**: Implement operations not in standard libraries
- **Performance**: 10-100x speedup over CPU for suitable workloads

## Thread/Block/Grid Model

CUDA's hierarchical parallelism model:
- **Thread**: Single execution unit (similar to CPU thread)
- **Block**: Group of threads (up to 1024) that can cooperate via shared memory
- **Grid**: Collection of blocks executing the same kernel

### Memory Hierarchy
1. **Registers**: Per-thread, fastest (~1 cycle latency)
2. **Shared Memory**: Per-block, fast (~5-30 cycles), explicitly managed
3. **L1/L2 Cache**: Automatic, hardware-managed
4. **Global Memory**: All threads, slow (~200-400 cycles), largest
5. **Constant Memory**: Read-only, cached, broadcast efficiently
6. **Texture Memory**: Cached, optimized for spatial locality

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Tuple, Optional

# Check CUDA availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    
    # Device properties
    props = torch.cuda.get_device_properties(0)
    print(f"\nDevice Properties:")
    print(f"  Compute capability: {props.major}.{props.minor}")
    print(f"  Total memory: {props.total_memory / 1e9:.2f} GB")
    print(f"  Multiprocessors: {props.multi_processor_count}")
    print(f"  Max threads per block: {props.max_threads_per_block}")
    print(f"  Max block dimensions: {props.max_threads_per_multi_processor}")
    print(f"  Warp size: {props.warp_size}")

## Writing Your First CUDA Kernel (via PyTorch)

While PyTorch abstracts CUDA, we can write custom CUDA kernels using:
1. **PyTorch C++ Extensions**: For production code
2. **CuPy**: Python interface to CUDA
3. **Numba**: JIT compilation with @cuda.jit decorator

Let's start with simple element-wise operations to understand memory access patterns.

In [ ]:
# Install numba if not available
# !pip install numba

try:
    from numba import cuda
    import math
    
    # Simple element-wise addition kernel
    @cuda.jit
    def add_kernel(a, b, c):
        """
        CUDA kernel for element-wise addition: c = a + b
        
        Each thread computes one element.
        Thread indexing: 
          - cuda.threadIdx.x: thread index within block (0 to blockDim.x-1)
          - cuda.blockIdx.x: block index within grid (0 to gridDim.x-1)
          - cuda.blockDim.x: number of threads per block
        """
        # Calculate global thread index
        idx = cuda.grid(1)  # Equivalent to: cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
        
        # Boundary check (important for non-multiple sizes)
        if idx < c.size:
            c[idx] = a[idx] + b[idx]
    
    # Test the kernel
    n = 1_000_000
    a = np.random.randn(n).astype(np.float32)
    b = np.random.randn(n).astype(np.float32)
    
    # Allocate device memory
    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_c = cuda.device_array(n, dtype=np.float32)
    
    # Configure kernel launch
    threads_per_block = 256
    blocks_per_grid = (n + threads_per_block - 1) // threads_per_block
    
    print(f"Launching kernel with {blocks_per_grid} blocks, {threads_per_block} threads/block")
    print(f"Total threads: {blocks_per_grid * threads_per_block}")
    
    # Launch kernel
    add_kernel[blocks_per_grid, threads_per_block](d_a, d_b, d_c)
    
    # Copy result back
    c = d_c.copy_to_host()
    
    # Verify correctness
    expected = a + b
    print(f"Max error: {np.abs(c - expected).max():.2e}")
    
    NUMBA_AVAILABLE = True
except ImportError:
    print("Numba not available. Install with: pip install numba")
    NUMBA_AVAILABLE = False

## Understanding Thread Indexing

The most important concept in CUDA: mapping threads to data.

In [ ]:
if NUMBA_AVAILABLE:
    # Kernel to demonstrate thread indexing
    @cuda.jit
    def index_demo_kernel(output):
        """
        Each thread writes its global index to output.
        Demonstrates 1D, 2D, and 3D indexing.
        """
        # 1D indexing
        idx_1d = cuda.grid(1)
        
        # 2D indexing (useful for matrices)
        # row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
        # col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
        
        if idx_1d < output.size:
            output[idx_1d] = idx_1d
    
    # Launch with small size to visualize
    n = 32
    d_out = cuda.device_array(n, dtype=np.int32)
    
    threads_per_block = 8
    blocks_per_grid = (n + threads_per_block - 1) // threads_per_block
    
    index_demo_kernel[blocks_per_grid, threads_per_block](d_out)
    result = d_out.copy_to_host()
    
    print(f"Thread indices (threads_per_block={threads_per_block}):")
    print(result)
    print(f"\nBlock boundaries: {list(range(0, n, threads_per_block))}")

## Memory Hierarchy Deep Dive

Understanding memory is crucial for CUDA performance.

In [ ]:
if NUMBA_AVAILABLE:
    # Shared memory example: parallel reduction
    @cuda.jit
    def sum_reduce_kernel(input_arr, output):
        """
        Parallel reduction using shared memory.
        Each block computes partial sum, stored in output.
        
        Shared memory:
        - Allocated per block
        - Much faster than global memory
        - Requires synchronization (cuda.syncthreads())
        """
        # Allocate shared memory (static size at compile time)
        sdata = cuda.shared.array(256, dtype=np.float32)
        
        tid = cuda.threadIdx.x
        idx = cuda.grid(1)
        
        # Load data from global to shared memory
        if idx < input_arr.size:
            sdata[tid] = input_arr[idx]
        else:
            sdata[tid] = 0.0
        
        # Synchronize threads in block
        cuda.syncthreads()
        
        # Parallel reduction in shared memory
        s = cuda.blockDim.x // 2
        while s > 0:
            if tid < s:
                sdata[tid] += sdata[tid + s]
            cuda.syncthreads()
            s //= 2
        
        # Thread 0 writes result for this block
        if tid == 0:
            cuda.atomic.add(output, 0, sdata[0])
    
    # Test reduction
    n = 10000
    arr = np.random.randn(n).astype(np.float32)
    
    d_arr = cuda.to_device(arr)
    d_out = cuda.to_device(np.zeros(1, dtype=np.float32))
    
    threads_per_block = 256
    blocks_per_grid = (n + threads_per_block - 1) // threads_per_block
    
    sum_reduce_kernel[blocks_per_grid, threads_per_block](d_arr, d_out)
    
    gpu_sum = d_out.copy_to_host()[0]
    cpu_sum = arr.sum()
    
    print(f"GPU sum: {gpu_sum:.6f}")
    print(f"CPU sum: {cpu_sum:.6f}")
    print(f"Error: {abs(gpu_sum - cpu_sum):.2e}")

## Matrix Multiplication Example

Classic CUDA example demonstrating:
- 2D thread blocks
- Shared memory for tiling
- Memory coalescing

**Naive vs Tiled Approach:**
- **Naive**: Each thread loads from global memory K times
- **Tiled**: Cooperative loading into shared memory, reuse across block

In [ ]:
if NUMBA_AVAILABLE:
    # Naive matrix multiplication
    @cuda.jit
    def matmul_naive(A, B, C):
        """
        Naive matrix multiplication: C = A @ B
        Each thread computes one element of C.
        
        Poor performance due to:
        - Redundant global memory loads
        - No memory reuse
        """
        row, col = cuda.grid(2)
        
        if row < C.shape[0] and col < C.shape[1]:
            tmp = 0.0
            for k in range(A.shape[1]):
                tmp += A[row, k] * B[k, col]
            C[row, col] = tmp
    
    # Tiled matrix multiplication using shared memory
    @cuda.jit
    def matmul_tiled(A, B, C):
        """
        Tiled matrix multiplication using shared memory.
        
        Key optimizations:
        1. Load tiles of A and B into shared memory
        2. Reuse loaded data across all threads in block
        3. Reduce global memory bandwidth by ~TILE_SIZE
        """
        TILE_SIZE = 16
        
        # Allocate shared memory for tiles
        sA = cuda.shared.array((16, 16), dtype=np.float32)
        sB = cuda.shared.array((16, 16), dtype=np.float32)
        
        row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
        col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
        
        tx = cuda.threadIdx.x
        ty = cuda.threadIdx.y
        
        tmp = 0.0
        
        # Loop over tiles
        for m in range((A.shape[1] + TILE_SIZE - 1) // TILE_SIZE):
            # Load tile into shared memory
            if row < A.shape[0] and (m * TILE_SIZE + tx) < A.shape[1]:
                sA[ty, tx] = A[row, m * TILE_SIZE + tx]
            else:
                sA[ty, tx] = 0.0
            
            if col < B.shape[1] and (m * TILE_SIZE + ty) < B.shape[0]:
                sB[ty, tx] = B[m * TILE_SIZE + ty, col]
            else:
                sB[ty, tx] = 0.0
            
            # Synchronize to ensure tile is loaded
            cuda.syncthreads()
            
            # Compute partial result for this tile
            for k in range(TILE_SIZE):
                tmp += sA[ty, k] * sB[k, tx]
            
            # Synchronize before loading next tile
            cuda.syncthreads()
        
        # Write result
        if row < C.shape[0] and col < C.shape[1]:
            C[row, col] = tmp
    
    # Benchmark both implementations
    def benchmark_matmul(M, N, K, use_tiled=False):
        A = np.random.randn(M, K).astype(np.float32)
        B = np.random.randn(K, N).astype(np.float32)
        C = np.zeros((M, N), dtype=np.float32)
        
        d_A = cuda.to_device(A)
        d_B = cuda.to_device(B)
        d_C = cuda.to_device(C)
        
        TILE_SIZE = 16
        threads_per_block = (TILE_SIZE, TILE_SIZE)
        blocks_per_grid_x = (N + TILE_SIZE - 1) // TILE_SIZE
        blocks_per_grid_y = (M + TILE_SIZE - 1) // TILE_SIZE
        blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)
        
        kernel = matmul_tiled if use_tiled else matmul_naive
        
        # Warmup
        kernel[blocks_per_grid, threads_per_block](d_A, d_B, d_C)
        cuda.synchronize()
        
        # Benchmark
        n_iter = 10
        start = time.time()
        for _ in range(n_iter):
            kernel[blocks_per_grid, threads_per_block](d_A, d_B, d_C)
        cuda.synchronize()
        elapsed = (time.time() - start) / n_iter
        
        # Verify correctness
        C_gpu = d_C.copy_to_host()
        C_cpu = A @ B
        error = np.abs(C_gpu - C_cpu).max()
        
        # Calculate GFLOPS
        flops = 2 * M * N * K  # multiply-add for each output element
        gflops = flops / elapsed / 1e9
        
        return elapsed * 1000, gflops, error
    
    print("Matrix Multiplication Benchmark:")
    sizes = [128, 256, 512, 1024]
    
    results = []
    for size in sizes:
        time_naive, gflops_naive, err_naive = benchmark_matmul(size, size, size, use_tiled=False)
        time_tiled, gflops_tiled, err_tiled = benchmark_matmul(size, size, size, use_tiled=True)
        
        results.append({
            'size': size,
            'naive_time': time_naive,
            'tiled_time': time_tiled,
            'naive_gflops': gflops_naive,
            'tiled_gflops': gflops_tiled,
            'speedup': time_naive / time_tiled
        })
        
        print(f"\nSize: {size}x{size}")
        print(f"  Naive: {time_naive:.2f}ms, {gflops_naive:.2f} GFLOPS")
        print(f"  Tiled: {time_tiled:.2f}ms, {gflops_tiled:.2f} GFLOPS")
        print(f"  Speedup: {time_naive/time_tiled:.2f}x")
        print(f"  Max error: {max(err_naive, err_tiled):.2e}")

In [ ]:
if NUMBA_AVAILABLE:
    # Visualize performance comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sizes = [r['size'] for r in results]
    naive_gflops = [r['naive_gflops'] for r in results]
    tiled_gflops = [r['tiled_gflops'] for r in results]
    speedups = [r['speedup'] for r in results]
    
    # GFLOPS comparison
    x = np.arange(len(sizes))
    width = 0.35
    axes[0].bar(x - width/2, naive_gflops, width, label='Naive', alpha=0.8)
    axes[0].bar(x + width/2, tiled_gflops, width, label='Tiled (Shared Memory)', alpha=0.8)
    axes[0].set_xlabel('Matrix Size')
    axes[0].set_ylabel('Performance (GFLOPS)')
    axes[0].set_title('Matrix Multiplication Performance')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'{s}x{s}' for s in sizes])
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Speedup
    axes[1].plot(sizes, speedups, 'o-', linewidth=2, markersize=8)
    axes[1].axhline(y=1, color='r', linestyle='--', alpha=0.5, label='Baseline')
    axes[1].set_xlabel('Matrix Size')
    axes[1].set_ylabel('Speedup (Tiled / Naive)')
    axes[1].set_title('Tiled Implementation Speedup')
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('cuda_matmul_performance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nKey Insights:")
    print("1. Tiled implementation consistently faster due to shared memory reuse")
    print("2. Speedup increases with matrix size (better amortization of overhead)")
    print("3. Shared memory reduces global memory bandwidth by ~16x (tile size)")
    print("4. Production libraries (cuBLAS) achieve 10-100x higher GFLOPS with more optimizations")

## Profiling with NVIDIA Tools

Understanding where time is spent:

### Command-line Tools
```bash
# Legacy profiler (deprecated)
nvprof python script.py

# Modern profilers
nsys profile python script.py  # System-wide profiling
ncu python script.py           # Kernel-level metrics
```

### PyTorch Profiler
Integrated profiling for PyTorch code.

In [ ]:
# PyTorch profiler example
if torch.cuda.is_available():
    from torch.profiler import profile, ProfilerActivity, record_function
    
    def model_forward(x):
        """Simple model for profiling."""
        with record_function("linear1"):
            x = torch.nn.functional.linear(x, torch.randn(512, 1024, device='cuda'))
        
        with record_function("activation"):
            x = torch.nn.functional.gelu(x)
        
        with record_function("linear2"):
            x = torch.nn.functional.linear(x, torch.randn(256, 512, device='cuda'))
        
        return x
    
    # Profile the model
    x = torch.randn(32, 1024, device='cuda')
    
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
        with_stack=True
    ) as prof:
        for _ in range(10):
            y = model_forward(x)
            torch.cuda.synchronize()
    
    # Print summary
    print("\nTop 10 operations by CUDA time:")
    print(prof.key_averages().table(
        sort_by="cuda_time_total",
        row_limit=10
    ))
    
    # Export to Chrome trace format
    # prof.export_chrome_trace("trace.json")
    # View at chrome://tracing

## Memory Bandwidth Analysis

GPU performance is often memory-bound, not compute-bound.

In [ ]:
if torch.cuda.is_available():
    def measure_bandwidth(size_mb=100, n_iter=100):
        """Measure GPU memory bandwidth."""
        size = size_mb * 1024 * 1024 // 4  # float32 elements
        
        x = torch.randn(size, device='cuda')
        y = torch.empty_like(x)
        
        # Warmup
        for _ in range(10):
            y.copy_(x)
        torch.cuda.synchronize()
        
        # Measure
        start = time.time()
        for _ in range(n_iter):
            y.copy_(x)
        torch.cuda.synchronize()
        elapsed = time.time() - start
        
        bytes_transferred = size * 4 * 2 * n_iter  # read + write
        bandwidth_gb_s = bytes_transferred / elapsed / 1e9
        
        return bandwidth_gb_s
    
    bandwidth = measure_bandwidth()
    print(f"Measured memory bandwidth: {bandwidth:.1f} GB/s")
    
    # Compare to theoretical peak
    # A100: ~1555 GB/s (PCIe: ~900 GB/s)
    # V100: ~900 GB/s
    # RTX 3090: ~936 GB/s
    print("\nTypical peak bandwidths:")
    print("  A100 (SXM): ~1555 GB/s")
    print("  V100: ~900 GB/s")
    print("  RTX 3090: ~936 GB/s")
    print("  RTX 4090: ~1008 GB/s")

## When to Write Custom CUDA Kernels

### Write Custom Kernels When:

1. **Operator Fusion**: Combine multiple operations to reduce memory traffic
   - Example: LayerNorm + residual + dropout in one kernel
   
2. **Custom Algorithms**: Implement specialized algorithms not in libraries
   - Example: Flash Attention, sparse attention patterns
   
3. **Memory Optimization**: Reduce memory footprint or bandwidth
   - Example: Gradient checkpointing, in-place operations
   
4. **Hardware-Specific Features**: Leverage new hardware capabilities
   - Example: Tensor Cores, async copy

### Don't Write Custom Kernels When:

1. **Standard Operations**: Use cuBLAS, cuDNN, etc.
   - They're heavily optimized by NVIDIA
   
2. **Small Bottlenecks**: Amdahl's Law applies
   - 2x speedup on 5% of runtime = 2.6% overall improvement
   
3. **Prototyping**: Use PyTorch/JAX for initial development
   - Optimize critical paths later
   
4. **Portability Concerns**: CUDA locks you to NVIDIA
   - Consider OpenCL, SYCL, or portable abstractions

In [ ]:
# Example: When fusion helps
if torch.cuda.is_available():
    def unfused_operations(x):
        """Three separate operations, each reads/writes global memory."""
        x = x * 2.0      # 2 memory ops (read x, write result)
        x = x + 1.0      # 2 memory ops
        x = torch.relu(x)  # 2 memory ops
        return x
    
    @torch.jit.script
    def fused_operations(x):
        """Fused via TorchScript JIT."""
        return torch.relu(x * 2.0 + 1.0)
    
    # Benchmark
    x = torch.randn(1024, 1024, device='cuda')
    
    # Warmup
    for _ in range(100):
        _ = unfused_operations(x)
        _ = fused_operations(x)
    torch.cuda.synchronize()
    
    # Measure unfused
    n_iter = 1000
    start = time.time()
    for _ in range(n_iter):
        y = unfused_operations(x)
    torch.cuda.synchronize()
    time_unfused = (time.time() - start) / n_iter * 1000
    
    # Measure fused
    start = time.time()
    for _ in range(n_iter):
        y = fused_operations(x)
    torch.cuda.synchronize()
    time_fused = (time.time() - start) / n_iter * 1000
    
    print(f"Unfused operations: {time_unfused:.3f} ms")
    print(f"Fused operations: {time_fused:.3f} ms")
    print(f"Speedup: {time_unfused / time_fused:.2f}x")
    print(f"\nMemory traffic reduction: ~3x (6 memory ops -> 2 memory ops)")

## CUDA Programming Best Practices

### 1. Memory Coalescing
- Adjacent threads access adjacent memory locations
- Maximize memory transaction efficiency
- Poor coalescing = wasted bandwidth

### 2. Occupancy
- Balance threads/blocks to hide latency
- More threads = better latency hiding
- Limited by registers, shared memory, warps/SM

### 3. Divergence
- Avoid if/else branches that differ between threads
- Threads in a warp execute in lockstep
- Divergence = serialization = slowdown

### 4. Bank Conflicts
- Shared memory is divided into banks
- Multiple threads accessing same bank = conflict
- Use padding to avoid conflicts

### 5. Tensor Cores
- Use mixed precision (FP16/BF16 compute, FP32 accumulate)
- Requires specific matrix sizes (multiples of 8/16)
- 8-16x speedup for matmul on modern GPUs

In [ ]:
# Demonstrate Tensor Core usage
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 7:
    # Tensor Cores require:
    # 1. FP16/BF16 input
    # 2. Matrix dimensions multiple of 8 (FP16) or 16 (some ops)
    # 3. CUDA compute capability >= 7.0 (Volta+)
    
    sizes = [128, 256, 512, 1024, 2048]
    
    print("Matrix Multiplication: FP32 vs FP16 (Tensor Cores)\n")
    
    for size in sizes:
        # FP32
        a_fp32 = torch.randn(size, size, device='cuda', dtype=torch.float32)
        b_fp32 = torch.randn(size, size, device='cuda', dtype=torch.float32)
        
        # Warmup
        torch.mm(a_fp32, b_fp32)
        torch.cuda.synchronize()
        
        start = time.time()
        for _ in range(100):
            c_fp32 = torch.mm(a_fp32, b_fp32)
        torch.cuda.synchronize()
        time_fp32 = (time.time() - start) / 100
        
        # FP16
        a_fp16 = a_fp32.half()
        b_fp16 = b_fp32.half()
        
        # Warmup
        torch.mm(a_fp16, b_fp16)
        torch.cuda.synchronize()
        
        start = time.time()
        for _ in range(100):
            c_fp16 = torch.mm(a_fp16, b_fp16)
        torch.cuda.synchronize()
        time_fp16 = (time.time() - start) / 100
        
        flops = 2 * size**3
        gflops_fp32 = flops / time_fp32 / 1e9
        gflops_fp16 = flops / time_fp16 / 1e9
        
        print(f"{size}x{size}:")
        print(f"  FP32: {time_fp32*1000:.2f}ms, {gflops_fp32:.1f} GFLOPS")
        print(f"  FP16: {time_fp16*1000:.2f}ms, {gflops_fp16:.1f} GFLOPS")
        print(f"  Speedup: {time_fp32/time_fp16:.2f}x\n")
else:
    print("Tensor Cores require CUDA compute capability >= 7.0 (Volta, Turing, Ampere, Hopper)")

## Summary

### Key Concepts
1. **Thread/Block/Grid hierarchy** for massive parallelism
2. **Memory hierarchy** (registers -> shared -> L1/L2 -> global)
3. **Coalescing** for efficient memory access
4. **Shared memory** for data reuse within blocks
5. **Occupancy** to hide memory latency

### Performance Hierarchy
1. **Use existing libraries** (cuBLAS, cuDNN) - 90% of the time
2. **Operator fusion** (TorchScript, XLA) - 9% of the time
3. **Custom kernels** (CUDA, Triton) - 1% of the time

### Next Steps
- **Notebook 51**: Triton - Python-like GPU programming
- **Notebook 52**: Custom attention kernels (Flash Attention)
- **Notebook 53**: Fused operations for transformers

### Resources
- [CUDA C++ Programming Guide](https://docs.nvidia.com/cuda/cuda-c-programming-guide/)
- [CUDA Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/)
- [Numba CUDA Documentation](https://numba.readthedocs.io/en/stable/cuda/)
- [PyTorch CUDA Semantics](https://pytorch.org/docs/stable/notes/cuda.html)